In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import random
import os

In [ ]:
# ----- 1. Böngésző indítása -----
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [ ]:
# ----- 2. Paraméterek -----
base_url = "https://www.zenga.hu/budapest+kiado+lakas+ar--201000+szoba-1-+alapterulet-33-?page={}"
num_pages = 10  # hány oldal
start_page = 1
all_links = []

for page in range(start_page, num_pages + start_page):
    url = base_url.format(page)
    driver.get(url)
    print(f"Nyitva oldal {page}, várj, míg betöltődik...")
    
    time.sleep(5)  # várakozás a betöltődésre (nagyobb oldalakhoz növelhető)
    
    # ----- 3. Hirdetés linkek kigyűjtése -----
    listings = driver.find_elements(By.CSS_SELECTOR, 'a[data-cy="advert-card-link"]')
    page_links = [a.get_attribute("href") for a in listings]
    print(f"Talált hirdetések ezen az oldalon: {len(page_links)}")
    
    all_links.extend(page_links)
    
    # Rövid szünet oldalanként
    time.sleep(1)

In [ ]:
# ----- 4. Eredmények mentése -----
output_path = "zenga_links_timi.csv"

# Ellenőrizzük, hogy létezik-e már a CSV fájl
if os.path.exists(output_path):
    # Ha létezik, betöltjük a meglévő adatokat
    existing_df = pd.read_csv(output_path)
    # Az új adatokat hozzáfűzzük
    updated_df = pd.concat([existing_df, pd.DataFrame(all_links, columns=['url'])], ignore_index=True)
    # Eltávolítjuk a duplikátumokat URL alapján
    updated_df = updated_df.drop_duplicates(subset=["url"], keep="last")
else:
    # Ha nem létezik, létrehozzuk az új DataFrame-t
    updated_df = pd.DataFrame(all_links, columns=['url'])

# Mentés CSV-be
updated_df.to_csv(output_path, index=False)
print(f"Adatok frissítve. Összesen {len(updated_df)} link van a CSV-ben.")

In [ ]:
# ----- 5. Böngésző bezárása -----
driver.quit()